# **Parte 1 – Scraping de decretos de emergencia**

### Responsable: Lucia Pulido

## 1. Importar librerías

In [180]:
import time
import re
import requests
import pandas as pd

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## 2. Revisar robots.txt

In [181]:
import requests

robots_url = "https://www.gob.pe/robots.txt"

respuesta_robots = requests.get(
    robots_url,
    headers={"User-Agent": "Mozilla/5.0"}
)

print(respuesta_robots.status_code)
print(respuesta_robots.text)

200
# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



#### Revisión de robots.txt

##### El archivo robots.txt de gob.pe establece restricciones para determinados robots de rastreo. Para el usuario general (User-agent: *) se prohíbe el acceso a /admin/ y a determinadas URLs que contienen los parámetros sheet. No se observa una prohibición general para la ruta /busquedas, por lo que esta ruta puede ser consultada.

##### Sin embargo, el archivo establece un Crawl-delay de 5 segundos para GPTBot y de 2 segundos para OAI-SearchBot. Aunque estas restricciones no correspondan directamente al navegador utilizado con Selenium, es recomendable realizar pausas entre las consultas para evitar enviar demasiadas solicitudes al servidor en poco tiempo y reducir la carga sobre el sitio.

## 3. Configuración de Selenium

In [182]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 20)

## 4. Scraping — Enero 2022

In [183]:
time.sleep(2)

url_enero = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-01-2022"
    "&hasta=31-01-2022"
)

driver.get(url_enero)

wait.until(
    EC.presence_of_all_elements_located(
        (By.TAG_NAME, "article")
    )
)

articles = driver.find_elements(By.TAG_NAME, "article")

print("Resultados encontrados:", len(articles))

Resultados encontrados: 8


#### Extraer datos - Enero 2022

In [184]:
datos_enero = []

for article in articles:

    enlace_elemento = article.find_element(
        By.CSS_SELECTOR,
        "h4 a"
    )

    numero = enlace_elemento.text.strip()

    fecha = article.find_element(
        By.XPATH,
        ".//span[contains(text(), 'Publicado:')]"
    ).text.strip()

    titulo = article.find_element(
        By.CSS_SELECTOR,
        "p"
    ).text.strip()

    enlace = enlace_elemento.get_attribute("href")

    if enlace.startswith("/"):
        enlace = "https://www.gob.pe" + enlace

    datos_enero.append({
        "mes": "enero",
        "numero": numero,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace
    })

df_enero = pd.DataFrame(datos_enero)

print("Resultados extraídos:", len(df_enero))

Resultados extraídos: 8


#### Tabla de documentos obtenidos — Enero 2022

In [185]:
df_enero

,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 011-2022-PCM,Publicado: 30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 010-2022-PCM,Publicado: 29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 009-2022-PCM,Publicado: 28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 004-2022-PCM,Publicado: 25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 007-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,enero,Decreto Supremo N.° 006-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,enero,Decreto Supremo N.° 005-2022-PCM,Publicado: 16 de enero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
7,enero,Decreto Supremo N.° 002-2022-PCM,Publicado: 6 de enero de 2022,Decreto Supremo que modifica el numeral 8.1 de...,https://www.gob.pe/institucion/pcm/normas-lega...


## 5. Scraping — Febrero 2022

In [186]:
time.sleep(2)

url_febrero = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-02-2022"
    "&hasta=28-02-2022"
)

driver.get(url_febrero)

wait.until(
    EC.presence_of_all_elements_located(
        (By.TAG_NAME, "article")
    )
)

articles = driver.find_elements(By.TAG_NAME, "article")

print("Resultados encontrados:", len(articles))

Resultados encontrados: 5


#### Extraer datos - Febrero 2022

In [187]:
datos_febrero = []

for article in articles:

    enlace_elemento = article.find_element(
        By.CSS_SELECTOR,
        "h4 a"
    )

    numero = enlace_elemento.text.strip()

    fecha = article.find_element(
        By.XPATH,
        ".//span[contains(text(), 'Publicado:')]"
    ).text.strip()

    titulo = article.find_element(
        By.CSS_SELECTOR,
        "p"
    ).text.strip()

    enlace = enlace_elemento.get_attribute("href")

    if enlace.startswith("/"):
        enlace = "https://www.gob.pe" + enlace

    datos_febrero.append({
        "mes": "febrero",
        "numero": numero,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace
    })

df_febrero = pd.DataFrame(datos_febrero)

print("Resultados extraídos:", len(df_febrero))

Resultados extraídos: 5


#### Tabla de documentos obtenidos — Febrero 2022

In [188]:
df_febrero

,mes,numero,fecha,titulo,enlace
0,febrero,Decreto Supremo N.° 016-2022-PCM,Publicado: 27 de febrero de 2022,Decreto Supremo que declara Estado de Emergenc...,https://www.gob.pe/institucion/pcm/normas-lega...
1,febrero,Decreto Supremo N.° 015-2022-PCM,Publicado: 12 de febrero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
2,febrero,Decreto Supremo N.° 014-2022-PCM,Publicado: 10 de febrero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,febrero,Decreto Supremo N.° 013-2022-PCM,Publicado: 10 de febrero de 2022,Prórroga del Estado de Emergencia en distritos...,https://www.gob.pe/institucion/pcm/normas-lega...
4,febrero,Decreto Supremo N.° 012-2022-PCM,Publicado: 2 de febrero de 2022,Declarar por termino de cuarenta y cinco (45) ...,https://www.gob.pe/institucion/pcm/normas-lega...


## 6. Scraping — Marzo 2022

In [189]:
time.sleep(2)

url_marzo = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-03-2022"
    "&hasta=31-03-2022"
)

driver.get(url_marzo)

wait.until(
    EC.presence_of_all_elements_located(
        (By.TAG_NAME, "article")
    )
)

articles = driver.find_elements(By.TAG_NAME, "article")

print("Resultados encontrados:", len(articles))

Resultados encontrados: 5


#### Extraer datos - Marzo 2022

In [190]:
datos_marzo = []

for article in articles:

    enlace_elemento = article.find_element(
        By.CSS_SELECTOR,
        "h4 a"
    )

    numero = enlace_elemento.text.strip()

    fecha = article.find_element(
        By.XPATH,
        ".//span[contains(text(), 'Publicado:')]"
    ).text.strip()

    titulo = article.find_element(
        By.CSS_SELECTOR,
        "p"
    ).text.strip()

    enlace = enlace_elemento.get_attribute("href")

    if enlace.startswith("/"):
        enlace = "https://www.gob.pe" + enlace

    datos_marzo.append({
        "mes": "marzo",
        "numero": numero,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace
    })

df_marzo = pd.DataFrame(datos_marzo)

print("Resultados extraídos:", len(df_marzo))

Resultados extraídos: 5


#### Tabla de documentos obtenidos — Marzo 2022

In [191]:
df_marzo

,mes,numero,fecha,titulo,enlace
0,marzo,Decreto Supremo N.° 029-2022-PCM,Publicado: 26 de marzo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,marzo,Decreto Supremo N.° 028-2022-PCM,Publicado: 26 de marzo de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
2,marzo,Decreto Supremo N.° 027-2022-PCM,Publicado: 26 de marzo de 2022,| Decreto Supremo que declara Estado de Emerge...,https://www.gob.pe/institucion/pcm/normas-lega...
3,marzo,Decreto Supremo N.° 030-2022-PCM,Publicado: 22 de marzo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,marzo,Resolución de Secretaría de Gestión Pública N....,Publicado: 11 de marzo de 2022,N° 002-2021-PCM/ SGP y adecúan de manera progr...,https://www.gob.pe/institucion/pcm/normas-lega...


## 7. Scraping — Abril 2022

In [192]:
time.sleep(2)

url_abril = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-04-2022"
    "&hasta=30-04-2022"
)

driver.get(url_abril)

wait.until(
    EC.presence_of_all_elements_located(
        (By.TAG_NAME, "article")
    )
)

articles = driver.find_elements(By.TAG_NAME, "article")

print("Resultados encontrados:", len(articles))

Resultados encontrados: 4


#### Extraer datos - Abril 2022

In [193]:
datos_abril = []

for article in articles:

    enlace_elemento = article.find_element(
        By.CSS_SELECTOR,
        "h4 a"
    )

    numero = enlace_elemento.text.strip()

    fecha = article.find_element(
        By.XPATH,
        ".//span[contains(text(), 'Publicado:')]"
    ).text.strip()

    titulo = article.find_element(
        By.CSS_SELECTOR,
        "p"
    ).text.strip()

    enlace = enlace_elemento.get_attribute("href")

    if enlace.startswith("/"):
        enlace = "https://www.gob.pe" + enlace

    datos_abril.append({
        "mes": "abril",
        "numero": numero,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace
    })

df_abril = pd.DataFrame(datos_abril)

print("Resultados extraídos:", len(df_abril))

Resultados extraídos: 4


#### Tabla de documentos obtenidos — Abril 2022

In [194]:
df_abril

,mes,numero,fecha,titulo,enlace
0,abril,Decreto Supremo N.° 041-2022-PCM,Publicado: 24 de abril de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,abril,Decreto Supremo N.° 035-2022-PCM,Publicado: 7 de abril de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
2,abril,Decreto Supremo N.° 034-2022-PCM,Publicado: 4 de abril de 2022,Decreto Supremo que prorroga el Estado de emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,abril,Decreto Supremo N.° 032-2022-PCM,Publicado: 1 de abril de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...


## 8. Scraping — Mayo 2022

In [195]:
time.sleep(2)

url_mayo = (
    "https://www.gob.pe/busquedas?"
    "contenido[]=normas"
    "&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones"
    "&sort_by=recent"
    "&desde=01-05-2022"
    "&hasta=31-05-2022"
)

driver.get(url_mayo)

wait.until(
    EC.presence_of_all_elements_located(
        (By.TAG_NAME, "article")
    )
)

articles = driver.find_elements(By.TAG_NAME, "article")

print("Resultados encontrados:", len(articles))

Resultados encontrados: 7


#### Extraer datos - Mayo 2022

In [196]:
datos_mayo = []

for article in articles:

    enlace_elemento = article.find_element(
        By.CSS_SELECTOR,
        "h4 a"
    )

    numero = enlace_elemento.text.strip()

    fecha = article.find_element(
        By.XPATH,
        ".//span[contains(text(), 'Publicado:')]"
    ).text.strip()

    titulo = article.find_element(
        By.CSS_SELECTOR,
        "p"
    ).text.strip()

    enlace = enlace_elemento.get_attribute("href")

    if enlace.startswith("/"):
        enlace = "https://www.gob.pe" + enlace

    datos_mayo.append({
        "mes": "mayo",
        "numero": numero,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace
    })

df_mayo = pd.DataFrame(datos_mayo)

print("Resultados extraídos:", len(df_mayo))


Resultados extraídos: 7


#### Tabla de documentos obtenidos — Mayo 2022

In [197]:
df_mayo

,mes,numero,fecha,titulo,enlace
0,mayo,Decreto Supremo N.° 060-2022-PCM,Publicado: 26 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,mayo,Decreto Supremo N.° 059-2022-PCM,Publicado: 26 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,mayo,Decreto Supremo N.° 058-2022-PCM,Publicado: 24 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,mayo,Decreto Supremo N.° 057-2022-PCM,Publicado: 20 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,mayo,Decreto Supremo N.° 056-2022-PCM,Publicado: 18 de mayo de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...
5,mayo,Decreto Supremo N.° 055-2022-PCM,Publicado: 18 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,mayo,Decreto Supremo N.° 054-2022-PCM,Publicado: 18 de mayo de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


## 9. Consolidación de resultados

In [198]:
df_todos = pd.concat(
    [df_enero, df_febrero, df_marzo, df_abril, df_mayo],
    ignore_index=True
)

print("Total de resultados:", len(df_todos))

Total de resultados: 29


### 9.1. Tabla conjunta de documentos — Enero a Mayo 2022

In [199]:
df_todos = pd.concat(
    [df_enero, df_febrero, df_marzo, df_abril, df_mayo],
    ignore_index=True
)

df_todos

,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 011-2022-PCM,Publicado: 30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 010-2022-PCM,Publicado: 29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 009-2022-PCM,Publicado: 28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 004-2022-PCM,Publicado: 25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 007-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,enero,Decreto Supremo N.° 006-2022-PCM,Publicado: 18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
6,enero,Decreto Supremo N.° 005-2022-PCM,Publicado: 16 de enero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
7,enero,Decreto Supremo N.° 002-2022-PCM,Publicado: 6 de enero de 2022,Decreto Supremo que modifica el numeral 8.1 de...,https://www.gob.pe/institucion/pcm/normas-lega...
8,febrero,Decreto Supremo N.° 016-2022-PCM,Publicado: 27 de febrero de 2022,Decreto Supremo que declara Estado de Emergenc...,https://www.gob.pe/institucion/pcm/normas-lega...
9,febrero,Decreto Supremo N.° 015-2022-PCM,Publicado: 12 de febrero de 2022,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...


## 10. Verificación de resultados por mes

In [200]:
orden_meses = [
    "enero",
    "febrero",
    "marzo",
    "abril",
    "mayo"
]

tabla_verificacion = (
    df_todos
    .groupby("mes")
    .size()
    .reindex(orden_meses)
    .reset_index(name="resultados_extraidos")
)

tabla_verificacion

,mes,resultados_extraidos
0,enero,8
1,febrero,5
2,marzo,5
3,abril,4
4,mayo,7


## 11. Eliminación de duplicados

In [201]:
antes = len(df_todos)

df_todos = df_todos.drop_duplicates(
    subset="enlace"
).reset_index(drop=True)

despues = len(df_todos)

print("Antes:", antes)
print("Después:", despues)
print("Duplicados eliminados:", antes - despues)

Antes: 29
Después: 29
Duplicados eliminados: 0


## 12. Filtrado de normas de la PCM

In [202]:
antes_pcm = len(df_todos)

df_todos = df_todos[
    df_todos["enlace"].str.contains(
        "/institucion/pcm/",
        na=False
    )
].copy()

eliminados_pcm = antes_pcm - len(df_todos)

print("Documentos eliminados:", eliminados_pcm)
print("Documentos restantes:", len(df_todos))

Documentos eliminados: 0
Documentos restantes: 29


## 13. Obtención de títulos completos

In [203]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

titulos_completos = []

for enlace in df_todos["enlace"]:

    respuesta = requests.get(
        enlace,
        headers=headers
    )

    sopa = BeautifulSoup(
        respuesta.text,
        "html.parser"
    )

    descripcion = sopa.find(
        "div",
        class_="description"
    )

    if descripcion:
        titulo_completo = descripcion.get_text(
            " ",
            strip=True
        )
    else:
        titulo_completo = ""

    titulos_completos.append(titulo_completo)

    time.sleep(1)

df_todos["titulo"] = titulos_completos

print("Títulos obtenidos:", len(titulos_completos))

Títulos obtenidos: 29


## 14. Identificación de normas sobre lluvias

In [204]:
df_todos["es_lluvia"] = df_todos["titulo"].str.lower().apply(
    lambda x: "lluvia" in x or "precipitaciones" in x
)

## 15. Clasificar tipo de norma

In [205]:
def clasificar_tipo(titulo):

    titulo = titulo.lower()

    if "prórroga" in titulo or "prorroga" in titulo:
        return "prorroga"

    elif "declara" in titulo:
        return "declara"

    else:
        return "otro"

df_todos["tipo"] = df_todos["titulo"].apply(
    clasificar_tipo
)

## 16. Clasificación del motivo

In [206]:
def clasificar_motivo(titulo):

    titulo = titulo.lower()

    if "peligro inminente" in titulo:
        return "peligro inminente"

    elif "impacto de daños" in titulo:
        return "impacto de daños"

    else:
        return "otro"

df_todos["motivo"] = df_todos["titulo"].apply(
    clasificar_motivo
)

## 17. Revisión de casos clasificados como "otro"

### Para revisar el tipo:

In [207]:
df_todos[df_todos["tipo"] == "otro"][
    ["numero", "titulo", "tipo"]
]

,numero,titulo,tipo
17,Resolución de Secretaría de Gestión Pública N....,Dejan sin efecto la Res. N° 002-2021-PCM/ SGP ...,otro


### Para revisar el motivo:

In [208]:
df_todos[df_todos["motivo"] == "otro"][
    ["numero", "titulo", "motivo"]
]

,numero,titulo,motivo
0,Decreto Supremo N.° 011-2022-PCM,Decreto Supremo que modifica el numeral 14.2 d...,otro
1,Decreto Supremo N.° 010-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,otro
3,Decreto Supremo N.° 004-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,otro
4,Decreto Supremo N.° 007-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,otro
5,Decreto Supremo N.° 006-2022-PCM,Decreto Supremo que prorroga el Estado de Emer...,otro
6,Decreto Supremo N.° 005-2022-PCM,Decreto Supremo que modifica el Decreto Suprem...,otro
7,Decreto Supremo N.° 002-2022-PCM,Decreto Supremo que modifica el numeral 8.1 de...,otro
8,Decreto Supremo N.° 016-2022-PCM,Decreto Supremo que declara Estado de Emergenc...,otro
9,Decreto Supremo N.° 015-2022-PCM,Decreto Supremo que modifica el Decreto Suprem...,otro
11,Decreto Supremo N.° 013-2022-PCM,Prórroga del Estado de Emergencia en distritos...,otro


## 18. Filtrado de normas sobre lluvias

In [209]:
df_lluvias = df_todos[
    df_todos["es_lluvia"] == True
].copy()

print(
    "Cantidad de normas sobre lluvias:",
    len(df_lluvias)
)

df_lluvias.head(5)

Cantidad de normas sobre lluvias: 1


,mes,numero,fecha,titulo,enlace,es_lluvia,tipo,motivo
21,abril,Decreto Supremo N.° 032-2022-PCM,Publicado: 1 de abril de 2022,Decreto Supremo que declara el Estado de Emerg...,https://www.gob.pe/institucion/pcm/normas-lega...,True,declara,impacto de daños


## 19. Identificación de departamentos

In [210]:
departamentos = [
    "Amazonas",
    "Áncash",
    "Apurímac",
    "Arequipa",
    "Ayacucho",
    "Cajamarca",
    "Callao",
    "Cusco",
    "Huancavelica",
    "Huánuco",
    "Ica",
    "Junín",
    "La Libertad",
    "Lambayeque",
    "Lima",
    "Loreto",
    "Madre de Dios",
    "Moquegua",
    "Pasco",
    "Piura",
    "Puno",
    "San Martín",
    "Tacna",
    "Tumbes",
    "Ucayali"
]

#### Función :

In [211]:
def identificar_departamentos(titulo):

    encontrados = []

    for departamento in departamentos:

        patron = r"\b" + re.escape(departamento) + r"\b"

        if re.search(patron, titulo):
            encontrados.append(departamento)

    return encontrados

df_lluvias["departamentos"] = df_lluvias["titulo"].apply(
    identificar_departamentos
)

df_lluvias[
    ["numero", "departamentos"]
]

,numero,departamentos
21,Decreto Supremo N.° 032-2022-PCM,"[Amazonas, Ayacucho, Piura]"


## 20. Declaratorias y prórrogas por departamento

In [212]:
tabla_departamentos = []

for departamento in departamentos:

    declaratorias = 0
    prorrogas = 0

    for _, fila in df_lluvias.iterrows():

        if departamento in fila["departamentos"]:

            if fila["tipo"] == "declara":
                declaratorias += 1

            elif fila["tipo"] == "prorroga":
                prorrogas += 1

    tabla_departamentos.append({
        "departamento": departamento,
        "declaratorias": declaratorias,
        "prorrogas": prorrogas
    })

df_departamentos = pd.DataFrame(
    tabla_departamentos
)

df_departamentos

,departamento,declaratorias,prorrogas
0,Amazonas,1,0
1,Áncash,0,0
2,Apurímac,0,0
3,Arequipa,0,0
4,Ayacucho,1,0
5,Cajamarca,0,0
6,Callao,0,0
7,Cusco,0,0
8,Huancavelica,0,0
9,Huánuco,0,0


## 21. Exportación de resultados

In [213]:
df_lluvias[
    ["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]
].to_csv(
    "datos/decretos_lluvias.csv",
    index=False,
    encoding="utf-8-sig"
)

In [214]:
df_departamentos.to_csv(
    "datos/decretos_por_departamento.csv",
    index=False,
    encoding="utf-8-sig"
)

In [215]:
print("Archivos guardados correctamente.")

Archivos guardados correctamente.


In [216]:
import os

print(os.getcwd())
print(os.listdir())

/Users/lucia.pt/Documents/GitHub/grupo13_python_basico/assignment_2
['datos', 'README.md', 'bitacora_ia.md', 'scraping_emergencias.ipynb', 'cruce_analisis.ipynb', 'api_lluvias.ipynb']


In [217]:
print(os.listdir("datos"))

['decretos_lluvias.csv', 'README.md', 'decretos_por_departamento.csv']
